# EP02｜數學語言：函數、指數與對數、Σ 符號

[![在 Colab 開啟](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/chainchen1103/llm-from-scratch/blob/main/notebooks/ep02/ep02_math_language.ipynb)

這份 notebook 對應網站上的 [EP02 圖文教學](https://chainchen1103.github.io/llm-from-scratch/episodes/ep02/)，段落和網站一樣，可以邊讀邊跑：

1. 工具一：函數（含畫函數圖）
2. 工具二：指數與 e
3. 工具三：對數
4. 工具四：Σ 與 Π
5. **本集重點：對數把乘法變成加法**
6. 動手做：從零寫出 `exp_series`、`e_by_compound`、`logsumexp`、`log_softmax`、`sequence_logprob`，並用測試檢查
7. 弄壞它：5 個「程式照跑、答案卻不對」的例子

這份 notebook 只用到 NumPy 和 matplotlib，Colab 已經內建，不用安裝任何東西；EP01 寫好的 `softmax` 也已經定義在下面，不用匯入。在自己的電腦上，用 EP01 建立的 `llm-labs` 專案開啟即可。動手做的最後一個檢查會和 PyTorch 比對：Colab 已經內建 PyTorch；沒裝 PyTorch 的話（EP01 步驟 7），那個檢查會自動跳過。

**使用方式**：從上到下依序執行（Shift + Enter）。標著「先猜猜看」的地方，先在心裡猜結果再執行。參考解答在最後，先自己寫寫看。

In [ ]:
# Colab 已經內建 NumPy 和 matplotlib，不用安裝；在自己的電腦上，請用 EP01 建立的 llm-labs 環境。
# %pip install -q numpy matplotlib

import math
import sys

import matplotlib
import numpy as np

print("Python", sys.version.split()[0], "| NumPy", np.__version__, "| matplotlib", matplotlib.__version__)

這份 notebook 用 NumPy 2.5 測試過。程式格裡的註解是說明，印出來的結果就在格子下方。有些運算會跳出 `RuntimeWarning` 警告（例如溢位），警告不是錯誤，程式會照常往下跑。

下面是 EP01 動手做寫好的 `softmax`（數值穩定版），這集很多地方會用到它：

$$
p_i = \frac{e^{z_i}}{\sum_j e^{z_j}}
$$

In [ ]:
def softmax(z, axis=-1):
    """沿著 axis 把分數 z 轉成機率。z 可以是任意形狀。"""
    z = np.asarray(z, dtype=float)
    shifted = z - z.max(axis=axis, keepdims=True)   # 每一列減掉自己的最大值
    e = np.exp(shifted)
    return e / e.sum(axis=axis, keepdims=True)      # 每一列除以自己的總和

## 1. 工具一：函數

**函數 function** 像一台機器：放一個東西進去，照固定的規則，出來一個東西。數學寫成 $f(x)$：$f$ 是函數的名字，$x$ 是輸入，$f(x)$（念「f of x」）是輸出。$f(x) = 2x + 1$ 的意思是「輸入乘 2 再加 1」，所以 $f(3) = 7$。

Python 的 `def` 就是函數（程式裡習慣叫「函式」），還能一次把整個 NumPy 陣列丟進去，每個元素各算一次（EP01 的向量化）：

In [ ]:
def f(x):
    return 2 * x + 1

print(f(3))
print(f(np.array([-2, -1, 0, 1, 2])))

### 畫函數圖

把每一組 $(x, f(x))$ 當成點連起來，就是函數圖。`matplotlib` 是畫圖套件，`plt` 是大家習慣的簡稱。圖上的文字先用英文，因為 matplotlib 預設的字型沒有中文，會變成方框。

In [ ]:
import matplotlib.pyplot as plt

x = np.linspace(-2, 2, 101)              # -2 到 2 平均切 101 個點
fig, ax = plt.subplots(figsize=(5, 4))   # 一張圖、一組座標軸
ax.plot(x, f(x), label="f(x) = 2x + 1")  # 101 個點連成線
ax.plot(x, x**2, label="g(x) = x^2")
ax.set(xlabel="x", ylabel="y")
ax.legend()                              # 顯示每條線的名字
ax.grid(True)                            # 加上格線
plt.show()                               # 在格子下方顯示這張圖

網站上的版本最後一行是 `fig.savefig(...)`，把圖另外存成檔案；在 notebook 裡用 `plt.show()` 直接顯示就好。之後畫損失曲線，就是把 $x$ 換成訓練步數、$y$ 換成損失。

### 輸入、輸出不一定是一個數

$f(x) = 2x + 1$ 是「一個數進、一個數出」。函數的輸入和輸出也可以是一整排數字。**先猜猜看**：第三行只把第 3 個分數從 3 改成 5，前兩個機率會變嗎？

In [ ]:
z = np.array([1.0, 2.0, 3.0])
print(np.sum(z))
print(softmax(z))
print(softmax(np.array([1.0, 2.0, 5.0])))

- 加總是「一排數字進、一個數出」：形狀 `(3,)` 進去，一個數出來。
- **Softmax 是「一排數字進、一排數字出」**：形狀 `(3,)` 進去，形狀 `(3,)` 出來。這種「一排數字」叫向量，現在先把它想成一維陣列，EP03 會正式介紹。
- 前兩個機率也跟著變了（0.090 → 0.017、0.245 → 0.047）。因為分母 $\sum_j e^{z_j}$ 用到了**所有**分數，Softmax 的每個輸出都由整個輸入向量決定。

### 合成函數與反函數

把一個函數的輸出，直接當成另一個函數的輸入，叫**合成函數 composite function**，寫成 $f(g(x))$：先把 $x$ 丟進 $g$，再把結果丟進 $f$。反過來，把結果復原成原本輸入的函數，叫**反函數 inverse function**，寫成 $f^{-1}$。$f(x) = 2x + 1$ 是「乘 2 再加 1」，倒過來「減 1 再除以 2」就是它的反函數 $f^{-1}(y) = (y - 1)/2$。

In [ ]:
def g(x):
    return x ** 2

def f_inv(y):
    return (y - 1) / 2

print(f(g(3)), g(f(3)))
print(f_inv(f(3)), f(f_inv(7)))

- **順序很重要**：$f(g(3)) = f(9) = 19$，$g(f(3)) = g(7) = 49$。
- **LLM 就是一個很長的合成函數**：「嵌入 → 第 1 層 → 第 2 層 → ⋯ → 第 24 層 → 反嵌入 → Softmax」（以 EP00 的 Qwen2.5-0.5B 為例），每一步的輸出都是下一步的輸入。
- $f^{-1}$ 念「f 的反函數」。這裡的 $-1$ **不是**次方，$f^{-1}(x)$ 不是 $1/f(x)$。
- 不是每個函數都能復原：$g(x) = x^2$ 把 $2$ 和 $-2$ 都變成 $4$。一路往上爬的函數（**遞增 increasing**：輸入越大、輸出越大）一定能復原。下一個工具的 $e^x$ 就是遞增的，它的反函數就是工具三的對數。

## 2. 工具二：指數與 e

### 次方與指數律

$a^n$ 念「a 的 n 次方」，意思是 $n$ 個 $a$ 相乘。$a$ 叫**底數 base**，$n$ 叫**指數 exponent**。次方的計算規則合稱**指數律 laws of exponents**（底數 $a$ 都是正數，$\cdot$ 和 $\times$ 一樣是乘號）：

| 規則 | 例子 | 為什麼 |
|---|---|---|
| $a^m \cdot a^n = a^{m+n}$ | $2^3 \cdot 2^2 = 2^5 = 32$ | 3 個 2 和 2 個 2 排在一起，是 5 個 2 |
| $a^m / a^n = a^{m-n}$ | $2^5 / 2^2 = 2^3$ | 上下約掉 2 個 2 |
| $(a^m)^n = a^{mn}$ | $(2^3)^2 = 2^6$ | 3 個 2 一組，共 2 組 |
| $a^0 = 1$ | $2^3 / 2^3 = 2^0 = 1$ | 套第二條 |
| $a^{-n} = 1/a^n$ | $2^{-3} = 1/8$ | $2^{-3} = 2^{0-3} = 2^0 / 2^3$ |
| $a^{1/2} = \sqrt{a}$ | $9^{1/2} = 3$ | $a^{1/2} \cdot a^{1/2} = a^1$ |

**最重要的是第一條：相乘＝指數相加**，後面的對數和本集重點都靠它。

In [ ]:
print(2**3, 2**3 * 2**2, 2**5)
print(2**0, 2**-3, 9**0.5)
print(2**10, 2**20, 2**30)
print(np.finfo(np.float64).max)     # float64 能存的最大數
print(np.float64(2.0) ** 1024)      # 會跳出 RuntimeWarning：溢位

第三行是**指數成長 exponential growth**：以 2 為底，指數每加 10，結果就約乘上 1000 倍（$2^{10} = 1024$）。`float64` 能存的最大數約 $1.8 \times 10^{308}$（`e+308` 是科學記號，代表乘上 $10^{308}$；這個 `e` 和下一節的自然常數 $e$ 無關），略小於 $2^{1024}$，所以最後一行溢位成 `inf`。

這裡特地用 NumPy 的 `float64`。Python 的整數沒有大小上限，Python 的小數則會直接報錯：

In [ ]:
print(len(str(2**1024)))      # 2**1024 是一個 309 位數
try:
    2.0 ** 1024
except OverflowError as e:
    print("OverflowError:", e)

### e 從哪裡來

**用複利找到它。** 存 1 元、年利率 100%，一年結算 $n$ 次、每次給 $\frac{1}{n}$ 的利息，錢每次都變成 $1 + \frac{1}{n}$ 倍，年底就有 $(1 + \frac{1}{n})^n$ 元。**先猜猜看**：結算得越頻繁，錢會無限變多嗎？

In [ ]:
for n in [1, 2, 12, 365, 1_000_000]:     # 每年、每半年、每月、每天、每百萬分之一年
    print(n, (1 + 1 / n) ** n)

不會。它越來越靠近 2.71828⋯ 這個數，叫**自然常數 e**，英文常叫 Euler's number。（`1_000_000` 就是 1000000，底線只是方便閱讀。網站上有可以拖動 $n$ 的互動元件。）

**另一種算出 $e$ 的方法：一直加下去。** $k!$ 念「k 的**階乘 factorial**」，是 $1 \times 2 \times \cdots \times k$，另外規定 $0! = 1$：

$$
e = \frac{1}{0!} + \frac{1}{1!} + \frac{1}{2!} + \frac{1}{3!} + \cdots = \sum_{k=0}^{\infty} \frac{1}{k!}
$$

像這樣無限多項加起來的式子，叫**級數 series**。`math.factorial(k)` 算 $k!$，下面讓 `k` 依序是 0、1、⋯、9，加 10 項：

In [ ]:
print(sum(1 / math.factorial(k) for k in range(10)))
print(math.e)

只加 10 項，小數點後 6 位就和 `math.e` 一樣了，比複利結算一百萬次還準。同樣的級數也能算 $e$ 的任何次方：

$$
e^x = \sum_{k=0}^{\infty} \frac{x^k}{k!} = 1 + x + \frac{x^2}{2!} + \frac{x^3}{3!} + \cdots
$$

第 6 節的動手做會用它從零寫出 `exp_series`，並和 `math.exp` 比對。

### 指數函數 eˣ

以 $e$ 為底數、$x$ 當輸入，就是**指數函數 exponential function** $e^x$，也寫成 $\exp(x)$。NumPy 是 `np.exp`：

In [ ]:
x = np.array([-2.0, -1.0, 0.0, 1.0, 2.0])
print(np.exp(x))
print(np.exp(2 + 3), np.exp(2) * np.exp(3))
print(np.exp(-1000.0), np.exp(1000.0))        # 會跳出 RuntimeWarning：溢位

四個性質：**永遠是正數**（再負也只是越來越靠近 0）、**遞增**、$e^0 = 1$、$e^{a+b} = e^a \cdot e^b$（第二行驗證了它）。最後一行是電腦的限制：數學上 $e^{-1000}$ 是一個正數，只是太小，電腦存成 `0.0`；$e^{1000}$ 太大，存成 `inf`。本集重點會處理這兩件事。

### 為什麼 Softmax 用指數函數

分數 logits 可以是任何數，包括負數。要把它變成機率，需要做到三件事：

1. **輸出一定是正的**。直接拿分數除以總和就不行：

In [ ]:
z = np.array([-1.0, 2.0])
print(z / z.sum())        # 「機率」變成 -1 和 2

2. **保持大小順序**：分數越高，機率越高。$e^x$ 是遞增的；換成 $x^2$ 就不行，$-3$ 和 $3$ 會變得一樣大。
3. **只看差距**：兩個機率的比值是 $\dfrac{p_i}{p_j} = \dfrac{e^{z_i}}{e^{z_j}} = e^{z_i - z_j}$。所以**分數差 1，機率就差 $e \approx 2.718$ 倍**，跟分數本身是多少無關。

EP00 的**溫度**是把分數除以 $T$ 再做 Softmax，分數的差距就變成原本的 $1/T$ 倍：

In [ ]:
z = np.array([1.0, 2.0, 3.0])
p = softmax(z)
print(p[1] / p[0], p[2] / p[1], np.e)
q = softmax(z / 2)                      # 溫度 T = 2
print(q)
print(q[1] / q[0], np.exp(0.5))

$T = 2$ 時，分數差 1 只讓機率差 $e^{0.5} \approx 1.65$ 倍，機率變得比較平均。（最後一行兩個數的最後一位不同，是 EP01 弄壞它 3 看過的小數誤差。）那 $2^z$、$10^z$ 呢？它們其實也符合這三點；為什麼偏偏用 $e$，學完對數再回答。

## 3. 工具三：對數

### 對數：指數的反函數

$b^x$ 是「給次方，求結果」。反過來「給結果，求次方」，就是**對數 logarithm**：$\log_b y$ 念「以 $b$ 為底，$y$ 的對數」，意思是「$b$ 的幾次方等於 $y$」。例如 $\log_{10} 1000 = 3$（因為 $10^3 = 1000$）、$\log_2 8 = 3$。

以 $e$ 為底的對數叫**自然對數 natural logarithm**，寫成 $\ln$。**機器學習的論文裡，沒寫底數的 $\log$ 幾乎都是 $\ln$**，NumPy 也一樣：`np.log` 就是 $\ln$，以 2 和 10 為底的另外叫 `np.log2`、`np.log10`。

In [ ]:
print(np.log10(1000), np.log2(8), np.log10(2))
print(np.log(np.e), np.log(1), np.log(np.e**2))

### 對數律：每一條都來自指數律

令 $a = e^m$、$b = e^n$，也就是 $m = \ln a$、$n = \ln b$，那麼 $ab = e^m \cdot e^n = e^{m+n}$，所以 $\ln(ab) = m + n = \ln a + \ln b$。**對數律 laws of logarithms** 每一條都是用同一招，把一條指數律翻過來（$a$、$b$ 都要是正數）：

| 對數律 | 來自哪一條指數律 |
|---|---|
| $\log(ab) = \log a + \log b$ | $e^m \cdot e^n = e^{m+n}$ |
| $\log(a/b) = \log a - \log b$ | $e^m / e^n = e^{m-n}$ |
| $\log(a^k) = k \log a$ | $(e^m)^k = e^{km}$ |
| $\log 1 = 0$ | $e^0 = 1$ |

**先猜猜看**：`np.log(0.0)` 和 `np.log(-1.0)` 會印出什麼？

In [ ]:
a, b = 2.0, 3.0
print(np.log(a * b), np.log(a) + np.log(b))
print(np.log(a / b), np.log(a) - np.log(b))
print(np.log(a ** 5), 5 * np.log(a))
print(np.log(np.array([1.0, 0.5, 0.293, 0.01])))
print(np.log(0.0))    # 會跳出 RuntimeWarning
print(np.log(-1.0))   # 會跳出 RuntimeWarning

- **第一條就是這集的主角：對數把乘法變成加法。**（第二行最後幾位不同，又是小數誤差。）
- 機率在 0 到 1 之間，所以**機率的對數都 ≤ 0**，機率越小越負。0.293 是 EP00 裡「今天天氣很」後面接「冷」的機率 29.3%。
- 沒有任何次方能讓 $e^x$ 剛好等於 0，所以 0 沒有對數；$x$ 越接近 0，$\ln x$ 就越往負的方向掉，NumPy 用 `-inf`（負無限大）表示。$e^x$ 永遠是正數，不可能等於 $-1$，所以 $\ln(-1)$ 也沒有答案，NumPy 給 `nan`（not a number）。

### ln x 是 eˣ 的鏡像，而且會把大數「壓扁」

反函數把輸入和輸出對調：點 $(a, b)$ 在 $y = e^x$ 的曲線上，點 $(b, a)$ 就在 $y = \ln x$ 上。所以兩條曲線以斜線 $y = x$ 為鏡子，互相對稱。這張圖就是網站動手做「加碼」要你寫成腳本、存成檔案的那一張，在 notebook 裡直接顯示就好：

In [ ]:
x_exp = np.linspace(-3, 2, 300)
x_log = np.linspace(np.exp(-3), np.exp(2), 300)   # ln x 的 x 範圍剛好是 e^x 的 y 範圍

fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(x_exp, np.exp(x_exp), label="y = e^x")
ax.plot(x_log, np.log(x_log), label="y = ln x")
ax.plot([-3, 7.5], [-3, 7.5], "--", color="gray", linewidth=1, label="y = x")

# 成對的點：(a, b) 在 e^x 上，(b, a) 就在 ln x 上
for a, b in [(0.0, 1.0), (1.0, np.e)]:
    ax.plot([a, b], [b, a], ":", color="black", linewidth=1)
    ax.plot([a], [b], "o", color="C0")
    ax.plot([b], [a], "o", color="C1")

ax.set_xlim(-3, 7.5)
ax.set_ylim(-3, 7.5)
ax.set_aspect("equal")                # 兩軸同比例，鏡像才看得出來
ax.axhline(0, color="black", linewidth=0.5)
ax.axvline(0, color="black", linewidth=0.5)
ax.set(xlabel="x", ylabel="y", title="e^x and ln x mirror each other across y = x")
ax.grid(True, alpha=0.3)
ax.legend(loc="upper left")
plt.show()

鏡像也說明了 $\ln x$ 為什麼爬得非常慢：$e^x$ 往上衝得多快，$\ln x$ 就往右爬得多慢。

In [ ]:
V = 151936                                # Qwen2.5-0.5B 的詞彙表大小
print(np.log(1_000_000))
print(np.log10(V), np.log2(V))
print(np.log(np.finfo(np.float64).max))
print(np.exp(709.0), np.exp(710.0))       # 會跳出 RuntimeWarning：溢位

- $\ln 1{,}000{,}000$ 只有約 13.8：對數會把很大的數「壓扁」成好處理的大小。
- $\log_{10} V \approx 5.18$，所以 $V$ 是 6 位數；$\log_2 V \approx 17.2$，所以 $2^{17} < V < 2^{18}$，要 18 個二進位數字才夠替每個 token 編號。EP07 會用 $\log_2$ 來量「資訊有多少」，單位就是**位元 bit**。
- `float64` 最大值的 $\ln$ 約是 709.78：這就是 EP01 說「$e^{710}$ 會溢位」的原因。

### 回到 EP01 的測試 4

EP01 的第 4 個測試用 `np.log` 從 EP00 溫度元件的 5 個機率反推出分數，這組分數直接做 Softmax，會剛好還原成原本的機率。令分數 $z_j = \ln p_j$，而所有 $p_j$ 加起來是 1：

$$
\text{softmax}(\ln p)_i = \frac{e^{\ln p_i}}{\sum_j e^{\ln p_j}} = \frac{p_i}{\sum_j p_j} = p_i
$$

加上溫度 $T$：用對數律第三條，$e^{(\ln p_i)/T} = p_i^{1/T}$。所以溫度 $T$ 的 Softmax 是「每個機率取 $1/T$ 次方，再除以總和」；$T = 2$ 就是每個機率開根號：

In [ ]:
p = np.array([0.62, 0.21, 0.09, 0.04, 0.04])
print(softmax(np.log(p)))
T = 2.0
print(softmax(np.log(p) / T))
print(p ** (1 / T) / np.sum(p ** (1 / T)))

**為什麼偏偏是 $e$？** 換成 2 當底數也可以。因為 $2^{z} = (e^{\ln 2})^{z} = e^{z \ln 2}$，用 2 當底數的 Softmax，等於先把分數乘上 $\ln 2 \approx 0.693$ 再做一般的 Softmax，也就是溫度 $T = 1/\ln 2 \approx 1.44$：

In [ ]:
z = np.array([1.0, 2.0, 3.0])
print(2.0**z / np.sum(2.0**z))     # 用 2 當底數的「Softmax」
print(softmax(z * np.log(2)))      # 分數先乘上 ln 2，再做一般的 Softmax

換底數只是換溫度，而分數是模型自己學出來的，會自己配合底數。大家都用 $e$，是因為 $e^x$ 的斜率最乾淨（EP04），而且它剛好和 `np.log`（$\ln$）配成一對。

## 4. 工具四：Σ 與 Π

### 怎麼念 Σ

$$
\sum_{i=1}^{n} x_i = x_1 + x_2 + \cdots + x_n
$$

**加總符號 summation notation** $\sum$ 念成「$x_i$，$i$ 從 1 加到 $n$」：下面的 $i = 1$ 是從哪裡開始，上面的 $n$ 是到哪裡結束（**包含** $n$），右邊的 $x_i$ 是每一項長什麼樣子。例如 $x = (3, 1, 4, 1, 5)$：$\sum_{i=1}^{5} x_i = 14$，$\sum_{i=2}^{4} x_i = 1 + 4 + 1 = 6$，平均 $\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i = 2.8$。

Softmax 分母的 $\sum_j$ 是「所有的 $j$ 都加」。分子的 $i$ 是「正在算的那一個」候選，分母要把**每一個**候選都加起來，所以換一個字母，就像兩層 for 迴圈不能用同一個變數名稱。

### Σ ＝ for 迴圈 ＝ `np.sum`

In [ ]:
x = np.array([3.0, 1.0, 4.0, 1.0, 5.0])   # 數學寫成 x_1, x_2, …, x_5
total = 0.0
for i in range(len(x)):                   # Python 的 i 是 0, 1, …, 4
    total = total + x[i]
print(total, np.sum(x), x.sum())
print(np.sum(x[1:4]))                     # Σ_{i=2}^{4} x_i
print(np.mean(x), np.sum(x) / len(x))

**數學的索引常從 1 開始，Python 從 0 開始**，翻譯時要小心：

| 數學 | Python |
|---|---|
| $x_1$ | `x[0]` |
| 第 $i$ 個 $x_i$ | `x[i - 1]` |
| 最後一個 $x_n$ | `x[n - 1]`，也就是 `x[-1]` |
| $\sum_{i=2}^{4} x_i$ | `np.sum(x[1:4])`（切片含頭不含尾） |

**練習**：把 $\sum_{i=1}^{n} x_i^2$ 寫成 for 迴圈和一行 NumPy，兩個結果要一樣。

In [ ]:
x = np.array([3.0, 1.0, 4.0, 1.0, 5.0])
total = 0.0
for i in range(len(x)):
    total = total + x[i] ** 2
print(total, np.sum(x ** 2))

### Π：連乘，以及一句話的機率

**連乘符號 product notation** $\prod_{i=1}^{n} x_i = x_1 \times x_2 \times \cdots \times x_n$，NumPy 是 `np.prod`。例如階乘就是連乘，$n! = \prod_{k=1}^{n} k$。

模型一次只預測下一個 token。讓它一個一個念出「今天／天／氣／很／冷」這 5 個 token，每一步都會給正確的下一個 token 一個機率 $p_1, \ldots, p_5$。整句話的機率是這 5 個數連乘：

$$
P(\text{今天天氣很冷}) = p_1 \times p_2 \times p_3 \times p_4 \times p_5 = \prod_{t=1}^{5} p_t
$$

把對數律第一條推廣到很多項，就得到 $\log \prod_{i=1}^{n} x_i = \sum_{i=1}^{n} \log x_i$：連乘之後取 log，等於各自取 log 再加總。

In [ ]:
print(np.prod([1, 2, 3, 4]), math.factorial(4))
print(np.log(np.prod([0.5, 0.2])), np.sum(np.log([0.5, 0.2])), np.log(0.1))

## 5. 本集重點：對數把乘法變成加法

### 連乘會小到存不下

一個 400 個 token 的回答，機率是 400 個小於 1 的數連乘。假設每一步的機率都是 0.1。**先猜猜看**：$n = 400$ 時，兩種算法各印出什麼？

In [ ]:
for n in [10, 100, 300, 400]:
    probs = np.full(n, 0.1)              # n 個 0.1
    print(n, np.prod(probs), np.sum(np.log(probs)))

- **直接連乘**：$0.1^{400} = 10^{-400}$，但 `float64` 能存的最小正數約 $5 \times 10^{-324}$，再小就變成 `0.0`。這叫**下溢 underflow**：溢位是太大存不下，下溢是太接近 0 存不下。
- **取 log 再加總**：400 個 $\ln 0.1 \approx -2.30$ 加起來是 $-921.03$，完全存得下。
- $\ln$ 是遞增函數，比 **log 機率 log probability** 的大小，就等於比機率的大小，什麼都沒損失。

這就是為什麼訓練時到處都是 $\log$：**模型計算、比較的幾乎都是 log 機率，而不是機率本身。**

### logsumexp：Softmax 分母的 log 版

Softmax 取 log 之後，分母會變成 $\log \sum_j e^{z_j}$。分數超過約 709.78 時，$e^{z_j}$ 就會溢位。解法和 EP01 一樣：先把最大的分數 $m = \max_j z_j$ 提出來：

$$
\log \sum_j e^{z_j} = \log\Big(e^{m} \sum_j e^{z_j - m}\Big) = m + \log \sum_j e^{z_j - m}
$$

每個分數先減掉最大值，再做 exp、加總、取 log，最後把最大值加回來。這個組合叫 **logsumexp**。

In [ ]:
z = np.array([1000.0, 1001.0, 1002.0])
print(np.log(np.sum(np.exp(z))))           # 會跳出 RuntimeWarning：溢位
m = np.max(z)
print(m + np.log(np.sum(np.exp(z - m))))

結果只比最大值 1002 大一點點，所以 logsumexp 可以想成「平滑版的最大值」：最大的那一項說了算。

### log-softmax：直接算 log 機率

$$
\log p_i = \log \frac{e^{z_i}}{\sum_j e^{z_j}} = z_i - \log \sum_j e^{z_j} = z_i - \text{logsumexp}(z)
$$

第 $i$ 個 log 機率，就是「自己的分數，減掉全部分數的 logsumexp」。這叫 **log-softmax**：

In [ ]:
z = np.array([1.0, 2.0, 3.0])
m = np.max(z)
lse = m + np.log(np.sum(np.exp(z - m)))
print(z - lse)
print(np.log(softmax(z)))

這個例子兩種寫法一樣。那為什麼不直接寫 `np.log(softmax(z))`？問題出在**差距**：只要某個分數比最大值小超過約 745，它的 $e^{z_j - m}$ 就會下溢成 0，Softmax 給它剛好 0 的機率，取 log 就得到 `-inf`。PyTorch 預設的 `float32` 範圍小得多，差距超過約 104 就會這樣：

In [ ]:
print(np.exp(-745.0), np.exp(-746.0))                          # float64：差距超過約 745 就下溢成 0
print(np.log(np.finfo(np.float64).smallest_subnormal))         # float64 能存的最小正數，取 ln
print(np.exp(np.float32(-103.0)), np.exp(np.float32(-104.0)))  # float32：差距超過約 104 就下溢成 0

log-softmax 不會：它只對 $\sum_j e^{z_j - m}$ 取 log，這個總和裡最大值那一項是 $e^0 = 1$，所以總和至少是 1。**訓練時要用 log-softmax，不要先做 Softmax 再取 log。** 弄壞它 2 會親眼看到。

## 6. 動手做：從零寫出 5 個函式

在下一格寫完這 5 個函式（網站上是寫在 `llm-labs/ep02/mathlang.py`），再執行下一格的檢查。

要求與提示：

1. **`exp_series`**：不用每次都算階乘。第 $k$ 項 $\frac{x^k}{k!}$ 等於第 $k-1$ 項乘上 $\frac{x}{k}$，所以從第 0 項的 $1$ 開始，每一圈乘上一個數就得到下一項。`e_by_compound` 一行就夠。
2. **`logsumexp`**：只用 NumPy，不寫 Python 迴圈。先用 `keepdims=True` 算出每一列自己的最大值（EP01 Softmax 的同一招），照上面的公式算完，最後依 `keepdims` 決定要不要用 `np.squeeze(out, axis=axis)` 拿掉那個長度 1 的維度。預設拿掉，和 `np.sum`、`torch.logsumexp` 一樣。
3. **`log_softmax`**：用你的 `logsumexp`。想清楚這裡的 `keepdims` 要設成什麼，廣播才會對（EP01 的「每一列減掉自己的最大值」）。
4. **`sequence_logprob`**：用 $\log \prod = \sum \log$，一行就夠。沿著最後一個維度加總：形狀 `(2, 400)` 代表 2 句話、每句 400 個 token，要回傳 2 個數。

In [ ]:
import numpy as np


def exp_series(x, n_terms=20):
    """用 e^x = Σ_{k=0}^{∞} x^k / k! 的前 n_terms 項（k = 0 … n_terms-1）近似 e^x。
    x 是一個數。只用純 Python，不能用 np.exp 或 math.exp。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 exp_series")


def e_by_compound(n):
    """回傳 (1 + 1/n)^n。n 越大，越接近 e。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 e_by_compound")


def logsumexp(z, axis=-1, keepdims=False):
    """數值穩定地算 log(Σ exp(z))，沿著 axis 做。z 可以是任意形狀。
    回傳形狀和 z.sum(axis=axis, keepdims=keepdims) 一樣：
    z 形狀 (4, 7)、axis=-1 → (4,)；keepdims=True → (4, 1)。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 logsumexp")


def log_softmax(z, axis=-1):
    """回傳 log(softmax(z))，但不能先算 softmax 再取 log。輸出形狀和 z 一樣。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 log_softmax")


def sequence_logprob(probs):
    """一串機率 p_1 … p_n，回傳 log(p_1 × … × p_n)，但不能先連乘。
    probs 形狀 (..., n)，回傳形狀 (...)：一維輸入就回傳一個數。"""
    # 你的實作
    raise NotImplementedError("還沒寫好 sequence_logprob")

下一格的 7 個檢查和網站上的 `tests/test_ep02_mathlang.py` 一樣，依序檢查：兩種算 $e$ 的方法；分數小時 `logsumexp` 和直接照公式算的結果一樣，各種 `axis`、`keepdims` 的形狀也對；分數很大時不溢位，而且每一列減的是**自己的**最大值；`np.exp(log_softmax(z))` 會還原成 `softmax(z)`；弄壞它 2 的情況不能出現 `-inf`；400 個 0.1 不會下溢；最後和 PyTorch 的 `torch.logsumexp`、`torch.log_softmax` 比對數值（沒裝 PyTorch 就跳過）。

改了上一格之後，要重新執行上一格，再執行這一格。

In [ ]:
def check_mathlang():
    """和網站上 tests/test_ep02_mathlang.py 一樣的 7 個測試。"""

    class Skipped(Exception):
        pass

    def two_ways_to_get_e():
        assert math.isclose(exp_series(1.0, 3), 2.5)      # 剛好 3 項：1 + 1 + 1/2
        assert math.isclose(exp_series(1.0, 20), math.e)
        assert math.isclose(exp_series(2.0, 30), math.exp(2.0))
        assert math.isclose(exp_series(-1.0, 30), math.exp(-1.0))
        assert e_by_compound(1) == 2.0
        assert math.isclose(e_by_compound(1_000_000), math.e, rel_tol=1e-6)

    def logsumexp_matches_naive_formula_on_small_inputs():
        rng = np.random.default_rng(0)
        z = rng.normal(size=(4, 7))
        naive = np.log(np.sum(np.exp(z), axis=-1))
        assert logsumexp(z).shape == (4,)
        assert np.allclose(logsumexp(z), naive)
        assert logsumexp(z, axis=0).shape == (7,)
        assert logsumexp(z, keepdims=True).shape == (4, 1)
        assert logsumexp(z[:1]).shape == (1,)

    def logsumexp_large_inputs():
        z = np.array([1000.0, 1001.0, 1002.0])
        expected = 1002.0 + math.log(1 + math.exp(-1) + math.exp(-2))
        assert np.isfinite(logsumexp(z))
        assert np.isclose(logsumexp(z), expected)
        rows = np.array([[0.0, 1.0, 2.0], [1000.0, 1001.0, 1002.0]])   # 每一列要減自己的最大值
        assert np.allclose(logsumexp(rows), [expected - 1000.0, expected])
        assert np.allclose(log_softmax(rows)[0], log_softmax(rows)[1])

    def exp_of_log_softmax_is_softmax():
        rng = np.random.default_rng(1)
        z = rng.normal(size=(3, 5)) * 10
        assert log_softmax(z).shape == z.shape
        assert np.allclose(np.exp(log_softmax(z)), softmax(z))
        assert np.allclose(np.exp(log_softmax(z, axis=0)), softmax(z, axis=0))

    def log_softmax_has_no_minus_inf():
        assert np.allclose(log_softmax(np.array([0.0, 1000.0])), [-1000.0, 0.0])

    def sequence_logprob_does_not_underflow():
        probs = np.full(400, 0.1)
        assert np.prod(probs) == 0.0          # 直接連乘：下溢成 0
        lp = sequence_logprob(probs)
        assert np.isfinite(lp)
        assert np.isclose(lp, 400 * math.log(0.1))
        assert sequence_logprob(np.full((2, 400), 0.1)).shape == (2,)   # 每一列是一句話

    def matches_pytorch():
        try:
            import torch
        except ImportError:
            raise Skipped("沒有安裝 PyTorch")
        rng = np.random.default_rng(2)
        z = rng.normal(size=(3, 5)) * 10
        t = torch.from_numpy(z)
        assert np.allclose(logsumexp(z), torch.logsumexp(t, dim=-1).numpy())
        assert np.allclose(log_softmax(z), torch.log_softmax(t, dim=-1).numpy())

    tests = [two_ways_to_get_e, logsumexp_matches_naive_formula_on_small_inputs, logsumexp_large_inputs,
             exp_of_log_softmax_is_softmax, log_softmax_has_no_minus_inf, sequence_logprob_does_not_underflow,
             matches_pytorch]
    passed = skipped = 0
    for t in tests:
        try:
            with np.errstate(all="ignore"):
                t()
            print("✓", t.__name__)
            passed += 1
        except NotImplementedError as e:
            print("…", t.__name__, f"（{e}）")
        except Skipped as e:
            print("-", t.__name__, f"跳過（{e}）")
            skipped += 1
        except Exception as e:
            print("✗", t.__name__, type(e).__name__, e)
    summary = f"{passed}/{len(tests)} 通過"
    if skipped:
        summary += f"，{skipped} 個跳過"
    print(summary)


check_mathlang()

## 7. 弄壞它

每一個都先**猜**結果，再執行，最後用自己的話解釋。完整的解釋在網站的「弄壞它」段落。

弄壞它 2 和 5 會用到你在第 6 節寫的 `log_softmax` 和 `exp_series`；還沒寫好的話，那一格會提醒你。也可以先執行最後的參考解答，再回來執行。

### 弄壞它 1：連乘下溢

第三個數學上和第二個完全相等，印出來也會一模一樣嗎？延伸的迴圈找出連乘變成 `0.0` 的臨界點（`dtype.__name__` 是型別的名字，例如 `float64`），`float32` 是 PyTorch 預設的小數型別。

In [ ]:
probs = np.full(400, 0.1)
print(np.prod(probs))
print(np.sum(np.log(probs)))
print(400 * np.log(0.1))

for dtype in [np.float64, np.float32]:     # 延伸：從第幾個 0.1 開始，連乘會變成 0？
    n = 1
    while np.prod(np.full(n, 0.1, dtype=dtype)) > 0:
        n += 1
    print(dtype.__name__, n)

後兩個差在最後一位：小數只能存有限的位數，每算一次都要四捨五入，留下一點點**捨入誤差 rounding error**。`np.sum` 要做 399 次加法，誤差累積起來；`400 * np.log(0.1)` 只做一次乘法。臨界點：`float64` 是第 324 個，`float32` 只要第 46 個。log 機率的加總完全沒有這個問題。

### 弄壞它 2：先 Softmax 再取 log

In [ ]:
z = np.array([0.0, 1000.0])
print(softmax(z))
print(np.log(softmax(z)))
try:
    print(log_softmax(z))        # 你在第 6 節寫的 log_softmax
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")

分數 0 比最大值 1000 小了 1000，$e^{-1000}$ 下溢成 0，Softmax 給了它剛好 0 的機率，`np.log(0.0)` 就是 `-inf`。真正的 log 機率是 $0 - \text{logsumexp}(z) \approx -1000$：很小，但是有限的數字。訓練時損失是 $-\log p$（$p$ 是正確答案的機率），如果正確答案剛好是這個 token，損失就是 `inf`，接下來整個訓練都會壞掉。

### 弄壞它 3：`np.log(100)` 是 2 嗎？

$10^2 = 100$，所以答案是 2？

In [ ]:
print(np.log(100))

`np.log` 是以 $e$ 為底的 $\ln$，$e^{4.605} \approx 100$。要以 10 為底，有三種寫法。第三種叫**換底 change of base**：$\log_{10} x = \ln x / \ln 10$。看到 $\log$，先確認底數；機器學習裡幾乎都是 $\ln$。

In [ ]:
print(np.log10(100), math.log10(100), np.log(100) / np.log(10))

### 弄壞它 4：Σ 差一個

想算 $\sum_{i=1}^{10} i = 1 + 2 + \cdots + 10$，和 $e$ 的前 10 項 $\sum_{k=0}^{9} \frac{1}{k!}$，寫成：

In [ ]:
print(sum(i for i in range(1, 10)))
print(sum(1 / math.factorial(k) for k in range(1, 10)))

正確答案應該是 55 和 2.718⋯，兩個都差了一項。第一個少了最後一項 10：`range` 含頭不含尾，$\sum_{i=1}^{n}$ 要寫成 `range(1, n + 1)`。第二個少了第一項 $\frac{1}{0!} = 1$：下標從 0 開始，`range` 也要從 0 開始。這種錯叫**差一錯誤 off-by-one error**：程式不會報錯，只是答案差一點，很難發現。改好之後：

In [ ]:
print(sum(i for i in range(1, 11)))
print(sum(1 / math.factorial(k) for k in range(10)))

### 弄壞它 5：用級數算 e⁻³⁰

用你寫的 `exp_series` 加 150 項（數學上已經非常準），和 `math.exp` 比：

In [ ]:
try:
    print(exp_series(-30.0, 150))    # 你在第 6 節寫的 exp_series
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")
print(math.exp(-30))

正確答案是 $e^{-30} \approx 9.36 \times 10^{-14}$。參考解答（每一圈 `term *= x / k`）算出來卻是 $6.1 \times 10^{-6}$，大了約 6500 萬倍。你的數字可能和這裡不一樣，甚至是負的，因為結果和運算順序有關。**試試看**：把參考解答（或你自己寫的版本）裡的 `term *= x / k` 改成 `term = term * x / k`（先乘 `x`，再除以 `k`），重新執行那一格和上面這格，參考解答會變成 $-3.07 \times 10^{-5}$。直接照公式寫又是另一個數字：

In [ ]:
print(sum((-30.0)**k / math.factorial(k) for k in range(150)))   # 直接照公式寫
print(30.0**30 / math.factorial(30))                             # 最大的一項

三種寫法數學上完全一樣，算出來卻差很多。$x = -30$ 時各項正負交錯，而且中間的項非常大（最大的一項約 $7.76 \times 10^{11}$）。`float64` 大約只有 16 位**有效數字 significant digits**，這麼大的數本身就帶著約 $10^{-4}$ 的捨入誤差；這些大項正負互相抵消，最後要剩下 $10^{-13}$ 這麼小的答案，留下來的幾乎全是誤差。

修法：$e^{-30} = 1 / e^{30}$。$x = 30$ 時每一項都是正的，沒有抵消：

In [ ]:
try:
    print(1 / exp_series(30.0, 150))
except NotImplementedError as e:
    print(f"{e}：先完成第 6 節的動手做，或執行最後的參考解答，再回來執行這格")
print(math.exp(-30))

這和 logsumexp、log-softmax 是同一個教訓：**數學上相等的寫法，在電腦上不一定一樣準。**

## 下一集

這集看到 $\sum_i a_i b_i$ 這種「對應位置相乘、再全部加起來」的寫法，它有自己的名字：內積。下一集 **EP03 向量與矩陣：內積、矩陣乘法、線性變換**，會正式介紹向量和矩陣。

## 參考解答

先自己寫，卡住再看。執行下一格會用參考解答取代第 6 節的函式，並重新跑一次檢查；之後可以回到上面，重新執行弄壞它 2 和 5。

In [ ]:
def exp_series(x, n_terms=20):
    """用 e^x = Σ_{k=0}^{∞} x^k / k! 的前 n_terms 項（k = 0 … n_terms-1）近似 e^x。
    x 是一個數。只用純 Python，不能用 np.exp 或 math.exp。"""
    total = 0.0
    term = 1.0                        # 第 0 項：x^0 / 0! = 1
    for k in range(1, n_terms + 1):
        total += term                 # 加上第 k-1 項
        term *= x / k                 # 第 k 項 = 第 k-1 項 × x / k，不用算階乘
    return total


def e_by_compound(n):
    """回傳 (1 + 1/n)^n。n 越大，越接近 e。"""
    return (1 + 1 / n) ** n


def logsumexp(z, axis=-1, keepdims=False):
    """數值穩定地算 log(Σ exp(z))，沿著 axis 做。z 可以是任意形狀。
    回傳形狀和 z.sum(axis=axis, keepdims=keepdims) 一樣：
    z 形狀 (4, 7)、axis=-1 → (4,)；keepdims=True → (4, 1)。"""
    z = np.asarray(z, dtype=float)
    m = z.max(axis=axis, keepdims=True)                              # 每一列自己的最大值
    out = m + np.log(np.sum(np.exp(z - m), axis=axis, keepdims=True))
    if not keepdims:
        out = np.squeeze(out, axis=axis)                             # 拿掉長度 1 的那個維度
    return out


def log_softmax(z, axis=-1):
    """回傳 log(softmax(z))，但不能先算 softmax 再取 log。輸出形狀和 z 一樣。"""
    z = np.asarray(z, dtype=float)
    return z - logsumexp(z, axis=axis, keepdims=True)                # keepdims=True 才能廣播回每一列


def sequence_logprob(probs):
    """一串機率 p_1 … p_n，回傳 log(p_1 × … × p_n)，但不能先連乘。
    probs 形狀 (..., n)，回傳形狀 (...)：一維輸入就回傳一個數。"""
    return np.sum(np.log(probs), axis=-1)


check_mathlang()